# Notebook title


## Introduction
This is a template for a model agnostic analysis notebook in the coastal commons cookbook. When developing or reviewing a notebook for inclusion in the coastal commons cookbook, please use this template as a guide for the notebook's structure. This notebook can be used by agentic agents to assist with coding or reviewing if you provide a link to it in your prompt.

The introduction section should include:
1. A short description of the purpose of the script
2. The methods used, including references and equations if appropriate
3. Any extra detail that a user needs to know to run the notebook (e.g. do they need to join a project to get access to a dataset)

Note that in this notebook we provide example code under each heading to demonstrate the format. This notebook is not meant to be executed.

## Initialisation
In this section, we load up or install libraries and start dask clients. The below cell is an example of common libraries and starting up a dask client. Adjust according to the notebooks requirements.



In [ ]:
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
from scipy.interpolate import griddata
client = Client(threads_per_worker = 1)
client

## Model agnostic translantions
In this section, we create a library to help translate between model - specific variables and netcdf structure and the model agnostic analysis being performed. The example below shows this for ROMS and MOM6. Adjust according to the notebooks requirements.


In [ ]:
model_args = {
              "roms": {"data_var": "zeta",
                       "lat_var": "lat_rho",
                       "lon_var": "lon_rho",
                       "time_var": 'ocean_time',
                       "z_var": "s_rho",
                       "z_inx": -1,
                       "file_name":'/g/data/yj27/users/cgk561/30yr_run_1993_2023_CK/output/roms_his_08[34567]*',

             },
              "mom6": { "data_var": "zos",
                       "lat_var": "yh",
                       "lon_var": "xh",
                       'time_var': 'time',
                       "file_name": "/g/data/av17/access-nri/rOM3/superaus/v20260619/expt08_outputs3/output*/access-om3.mom6.2d.zos.1day.mean*.nc",
 
              }
}

## Loading in the data
This section sets the paths to the data and opens the files, usually using x-array.
This section should also include setting parameters used in the analysis. For example, setting g (gravitational constant) or bounding boxes for the analysis. User configurable options should all be set in this section. Some examples of uploading datasets in a model-agnostic way and setting constants for a user-configurable bounding box are shown below.

In [ ]:
# Reading the data. 
models = ['roms', 'mom6']
ssh = {}
for model in models:
    varname = model_args[model]['data_var']
    files = model_args[model]['file_name']
    time_var = model_args[model]['time_var']
    lat_var= model_args[model]['lat_var']
    lon_var= model_args[model]['lon_var']
    time_var = model_args[model]['time_var']
    ssh[model] = xr.open_mfdataset(files, parallel=True, decode_timedelta=True,chunks={time_var: 1, lat_var: 'auto', lon_var: 'auto'} )[varname]
    ssh[model] = ssh[model].drop_duplicates(time_var, keep='first')

lon_min = 145
lon_max = 165
lat_min = -42
lat_max = -24

## Subsetting for a region
This section is optional. For larger domains and computationally expensive analysis, subsetting for a region of interest can help speed up the analysis. This subsetting should be done here. An example on how to do this in a model-agnostic analysis is shown below.

In [ ]:
#Subsettng by lat and lon 
for model in models:
    lat_var= model_args[model]['lat_var']
    lon_var= model_args[model]['lon_var']
    mask = ( (ssh[model][lat_var] < lat_max) & (ssh[model][lat_var] > lat_min)  & (ssh[model][lon_var] >lon_min) &  (ssh[model][lon_var]<lon_max)).compute()
    ssh[model] = ssh[model].where(mask, drop=True)

## Functions
Any functions needed for the analysis should be included here, along with a description on their purpose and use.
The purpose of these notebooks is to help people with a range of coding ability (novice through to expert) and the expectation is that users should be able to easily modify this notebook to suit their needs. A small number of larger functions that perform a specific analysis can be useful in some cases for improving readability if the analysis has several complicated steps. However, outside of this, functions should be kept to a minimum. Small helper functions should be avoided as they can make the code complicated for a novice coder to modify for their needs. There are no examples provided here as these will be very specific to the analysis being performed

## Analysis
The bulk of the analysis on the data is performed here. No examples provided as this will be very specific to the analysis being performed

## Plotting
Plots, movies and other outputs are done in this section. An example is shown below for model agnostic plots. Adjust according to the notebooks requirements.

In [ ]:
ncols = len(model_args)
fig, ax = plt.subplots(ncols=ncols, figsize=(ncols*5, 4))
n = 0
for model in models:
    lat_var= model_args[model]['lat_var']
    lon_var= model_args[model]['lon_var']
    time_var = model_args[model]['time_var']
    ssh[model].isel({time_var:0}).plot(ax=ax[n], x=lon_var, y=lat_var)
    ax[n].set_title(model + ' SLA')
    n = n+1